In [2]:
# Librerías
library(dplyr)
library(tidyr)
library(tibble)
library(readr)
library(shazam)


Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union


Loading required package: ggplot2

Warning message:
"replacing previous import 'S4Arrays::makeNindexFromArrayViewport' by 'DelayedArray::makeNindexFromArrayViewport' when loading 'SummarizedExperiment'"


To cite the SHazaM package in publications, please use:

  Gupta N, Vander Heiden J, Uduman M, Gadala-Maria D, Yaari G,
  Kleinstein S (2015). "Change-O: a toolkit for analyzing large-scale B
  cell immunoglobulin repertoire sequencing data." _Bioinformatics_,
  1-3. doi:10.1093/bioinformatics/btv359
  <https://doi.org/10.1093/bioinformatics/btv359>.

To cite the selection analysis methods, please use:

  Yaari G, Uduman M, Kleinstein S (2012). "Quantifying selection in
  high-throughput Immunoglobulin sequencing data sets." _Nucleic acids
  research_, *40*(17), e134. doi:10.1093/nar/gks

In [4]:
# ---------- LIBRERÍAS ----------
library(dplyr)
library(tidyr)
library(readr)

# ---------- RUTAS ----------
raiz    <- "/Users/catg/Desktop/SOFIAC/Gitsofia/tesisbioinf-sofia"
archivo <- file.path(raiz, "data/output/repertorio_D_insilico_100_seqs_clone-pass.tsv")
nombre  <- sub("_clone-pass\\.tsv$", "", basename(archivo))

dir_res <- file.path(raiz, "results/signatures/sin_colapsar")
dir.create(dir_res, showWarnings = FALSE, recursive = TRUE)

# ---------- LEER ----------
db <- read_tsv(archivo, show_col_types = FALSE) %>%
  filter(!is.na(germline_alignment), germline_alignment != "")

seqs  <- toupper(db$sequence_alignment)   # secuencias con mutaciones
germs <- toupper(db$germline_alignment)   # germline (original)
L <- min(max(nchar(seqs)), max(nchar(germs)))

# ---------- RECORRER POSICIONES ----------
lista <- list()

for (j in 3:(L - 2)) {
  motif         <- substr(germs, j - 2, j + 2)   # 5-mer de la germline
  base_original <- substr(germs, j, j)
  base_nueva    <- substr(seqs, j, j)

  hay_mut <- grepl("^[ACGT]{5}$", motif) &
             base_nueva %in% c("A", "C", "G", "T") &
             base_nueva != base_original

  if (any(hay_mut)) {
    lista[[j]] <- data.frame(
      motif    = motif[hay_mut],
      mutacion = paste0(base_original[hay_mut], ">", base_nueva[hay_mut])
    )
  }
}

# ---------- TABLA FINAL ----------
tabla <- bind_rows(lista) %>%
  count(motif, mutacion, name = "conteo") %>%
  mutate(muestra = nombre, .before = 1) %>%
  arrange(desc(conteo))

head(tabla)

write_csv(tabla, file.path(dir_res, paste0(nombre, "_motif_mutacion.csv")))

,muestra,motif,mutacion,conteo
,<chr>,<chr>,<chr>,<int>
1,repertorio_D_insilico_100_seqs,ACTTT,T>C,1
2,repertorio_D_insilico_100_seqs,CTCCT,C>A,1
3,repertorio_D_insilico_100_seqs,GTTCG,T>A,1


In [6]:
library(dplyr)
library(readr)

raiz    <- "/Users/catg/Desktop/SOFIAC/Gitsofia/tesisbioinf-sofia"
dir_res <- file.path(raiz, "results/signatures/sin_colapsar")
dir.create(dir_res, showWarnings = FALSE, recursive = TRUE)

# ---------- 11 profundidades del repertorio D ----------
profundidades <- 100 * 2^(0:10)
profundidades   # 100 200 400 ... 102400

todos <- list.files(file.path(raiz, "data/output"),
                    pattern = "^repertorio_D_insilico_.*_clone-pass\\.tsv$",
                    full.names = TRUE, recursive = TRUE)

# Quedarse solo con las profundidades de la lista
prof_archivo <- as.integer(sub(".*insilico_([0-9]+)_seqs.*", "\\1", basename(todos)))
archivos <- todos[prof_archivo %in% profundidades]
archivos <- archivos[order(prof_archivo[prof_archivo %in% profundidades])]

cat("Archivos encontrados:", length(archivos), "de", length(profundidades), "\n")
print(basename(archivos))

# Avisa cuáles faltan
faltan <- setdiff(profundidades, prof_archivo)
if (length(faltan) > 0) cat("Faltan profundidades:", faltan, "\n")

# ---------- FUNCIÓN: un archivo -> tabla motif, mutacion, conteo ----------
procesar <- function(archivo) {
  nombre <- sub("_clone-pass\\.tsv$", "", basename(archivo))

  db <- read_tsv(archivo, show_col_types = FALSE) %>%
    filter(!is.na(germline_alignment), germline_alignment != "")

  seqs  <- toupper(db$sequence_alignment)
  germs <- toupper(db$germline_alignment)
  L <- min(max(nchar(seqs)), max(nchar(germs)))

  lista <- list()
  for (j in 3:(L - 2)) {
    motif         <- substr(germs, j - 2, j + 2)
    base_original <- substr(germs, j, j)
    base_nueva    <- substr(seqs, j, j)

    hay_mut <- grepl("^[ACGT]{5}$", motif) &
               base_nueva %in% c("A", "C", "G", "T") &
               base_nueva != base_original

    if (any(hay_mut)) {
      lista[[j]] <- data.frame(
        motif    = motif[hay_mut],
        mutacion = paste0(base_original[hay_mut], ">", base_nueva[hay_mut])
      )
    }
  }

  cat(nombre, "| secuencias:", nrow(db), "\n")

  bind_rows(lista) %>%
    count(motif, mutacion, name = "conteo") %>%
    mutate(muestra = nombre, n_seqs = nrow(db), .before = 1)
}

# ---------- CORRER TODAS ----------
tabla_D <- bind_rows(lapply(archivos, procesar)) %>%
  mutate(
    escenario   = "repertorio_D",
    profundidad = as.integer(sub(".*insilico_([0-9]+)_seqs.*", "\\1", muestra))
  )

# Resumen: mutaciones totales por profundidad
tabla_D %>%
  group_by(profundidad) %>%
  summarise(mutaciones_totales = sum(conteo),
            motivos_distintos = n_distinct(motif))

write_csv(tabla_D, file.path(dir_res, "tabla_motif_mutacion_repertorio_D.csv"))

[1]    100    200    400    800   1600   3200   6400  12800  25600  51200
[11] 102400

Archivos encontrados: 11 de 11 
 [1] "repertorio_D_insilico_100_seqs_clone-pass.tsv"   
 [2] "repertorio_D_insilico_200_seqs_clone-pass.tsv"   
 [3] "repertorio_D_insilico_400_seqs_clone-pass.tsv"   
 [4] "repertorio_D_insilico_800_seqs_clone-pass.tsv"   
 [5] "repertorio_D_insilico_1600_seqs_clone-pass.tsv"  
 [6] "repertorio_D_insilico_3200_seqs_clone-pass.tsv"  
 [7] "repertorio_D_insilico_6400_seqs_clone-pass.tsv"  
 [8] "repertorio_D_insilico_12800_seqs_clone-pass.tsv" 
 [9] "repertorio_D_insilico_25600_seqs_clone-pass.tsv" 
[10] "repertorio_D_insilico_51200_seqs_clone-pass.tsv" 
[11] "repertorio_D_insilico_102400_seqs_clone-pass.tsv"
